# WhisperWatch retraining on live Whisper outputs

This experiment asks whether training the same boosted-tree model on fresh Whisper measurements improves review priority on English meetings. It compares against the original frozen model and native Whisper confidence. Open this notebook in a hosted **Google Colab T4 GPU** session. The candidate is for batch review ranking and is saved separately from the current model.

## 1. Install the Colab dependencies

The Whisper checkpoint, decoder settings and package versions match the earlier live run. Keep this session connected until you download the results.

In [ ]:
%pip install -q "pandas==2.2.3" "pyarrow>=23.0.1" "scikit-learn==1.8.0" "joblib==1.5.3" "faster-whisper==1.2.1" "ctranslate2==4.9.0" "av==16.0.1" "huggingface-hub==1.32.0" "nvidia-cublas-cu12" "nvidia-cudnn-cu12==9.*"

## 2. Upload the frozen plan and code

Upload `whisperwatch-live-retraining-code.zip`. The sample was drawn with seed 20261114 from AMI individual-headset recordings, using only segment IDs, meeting and speaker identifiers, and a metadata duration of 2–25 seconds. The models and their scores did not affect selection. Training and holdout use AMI's published train and test partitions, with a further check that speakers, meetings and related meetings never cross the split. The holdout is distinct from the official Whispered test set.

AMI is licensed CC BY 4.0. Its meeting speech is closer to conversational subtitles than read audiobooks, though individual headset microphones and annotated utterance boundaries differ from many real subtitle jobs. We cannot establish whether Whisper itself saw any recordings in pretraining.

[AMI dataset card and attribution](https://huggingface.co/datasets/edinburghcstr/ami)

The 750 holdout clips come from only four meeting groups. The paired interval and the leave-one-group-out results are exploratory. A strong result here would need to repeat on a separate set with more independent meetings.

In [ ]:
import json
import sys
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

from google.colab import files
from IPython.display import display

project = Path("/content/whisperwatch_live_retraining")
project.mkdir(exist_ok=True)

def upload_zip(prompt):
    print(prompt)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose one ZIP.")
    return BytesIO(next(iter(uploaded.values())))

with ZipFile(upload_zip("Upload whisperwatch-live-retraining-code.zip.")) as bundle:
    for member in bundle.namelist():
        directory, filename = member.split("/")
        if (directory == "src" and filename.endswith(".py")) or member in {"plan/preflight.json", "plan/manifest.csv"}:
            path = project / member
            path.parent.mkdir(exist_ok=True)
            path.write_bytes(bundle.read(member))

sys.path.insert(0, str(project))
plan = project / "plan"
output = project / "results"
print((plan / "preflight.json").read_text())


## 3. Transcribe, train and evaluate

Upload the original `whisperwatch-boosted-results.zip`. Colab downloads the pinned AMI shards listed above and, if uncached, about 3.1 GB of Whisper weights. The T4 transcribes the frozen clips in a separate GPU process. A quiet period after audio retrieval is normal. The notebook records retrieval and transcription failures and scores all methods on the same eligible holdout.

The candidate uses the original ten inputs and boosted-tree settings. Train labels come from the word error rate (WER) of fresh transcripts against the human references. High error means WER strictly above 20%. References are never classifier features. The candidate is saved and reloaded before holdout labels are computed. Both hybrids use the same average of boosted and native confidence percentile ranks, with no weight or threshold search. Each ranking gets ceil(20% of evaluated held-out clips) review slots. Clip ID breaks ties. The paired bootstrap re-ranks after sampling whole connected groups of meetings that share speakers.

The primary comparison is new hybrid versus native confidence. The comparison with the original hybrid tests whether this new training setup helped. It changes both the domain and the training-data size, so the experiment cannot isolate the effect of fresh features alone. A promising outcome is a practical recall gain of roughly 3–5 percentage points or more, with the paired interval above zero and enough eligible high-error clips and meeting groups to interpret it.

In [ ]:
from src.live_retraining import run

original = upload_zip("Upload the original whisperwatch-boosted-results.zip.")
comparison, intervals, runtime = run(plan, original, output)
display(comparison)
display(intervals)
print(json.dumps({key: value for key, value in runtime.items() if key != "settings"}, indent=2))


## 4. Download the results

The results ZIP contains the new candidate, frozen manifest, scores, confidence comparisons, runtime and failure logs, and examples of high-error clips each method uniquely found. Audio and source shards stay in the runtime. Keep this ZIP outside Git.

In [ ]:
download = Path("/content/whisperwatch-live-retraining-results.zip")
with ZipFile(download, "w", ZIP_DEFLATED) as bundle:
    for item in sorted(output.iterdir()):
        if item.suffix in {".csv", ".json", ".md", ".joblib"} and item.name != "live_input.json":
            bundle.write(item, item.name)
    bundle.write(plan / "preflight.json", "preflight.json")
files.download(str(download))
